# syft-ingest API Exploration

In [1]:
import syft_ingest as si
from syft_ingest import FetchConfig, async_gather, gather

## 1. Simplified gather() API — YouTube video

In [ ]:
# Simplest case: just platform + URLs
corpus = si.gather(
    "youtube", ["https://www.youtube.com/watch?v=zY2dAK-pMPI&t=11s"]
)  # Andrew Trask: talk

print(f"Items fetched: {len(corpus.all_items())}")
if corpus.all_items():
    item = corpus.all_items()[0]
    print(f"  Title: {item.title}")
    print(f"  Author: {item.author}")

In [ ]:
corpus.export("./data/youtube.jsonl")

## 2. Channel enumeration with config

In [ ]:
# With config options passed as kwargs
corpus = si.gather(
    "youtube",
    ["https://www.youtube.com/@iamtrask"],
    playlistend=3,  # Cap at 3 videos to keep it fast
)

print(f"Videos found: {len(corpus.all_items())}")
for item in corpus.all_items():
    print(f"  [{item.published_at}] {item.title}")

In [ ]:
corpus.export("./data/youtube2.jsonl")

## 3. Facebook scraping

Requires `BRIGHTDATA_API_TOKEN`. Triggers a real scrape job and polls until done (~60-120s).

In [2]:
import os

token = os.getenv("BRIGHTDATA_API_TOKEN")
print(
    f"BRIGHTDATA_API_TOKEN : {token[:3] + '...' + token[-3:] if token else 'NOT SET -- BrightData cells will fail'}"
)

BRIGHTDATA_API_TOKEN : 7a5...1d5


In [5]:
corpus = await async_gather(
    "facebook",
    ["https://www.facebook.com/profile.php?id=61583734012155"],
    author="Syft Influencer Test",
    post_limit=5,
)

print(f"Items fetched: {len(corpus.all_items())}")
print(f"Author: {corpus.person}")

if corpus.all_items():
    item = corpus.all_items()[0]
    print("\nFirst item:")
    print(f"  Type: {type(item).__name__}")
    print(f"  Title: {item.title}")
    print(f"  Text: {(item.text or '')[:120]}")

2026-04-13 21:16:31.917 | INFO     | syft_ingest.sources.brightdata:fetch_async:116 - Fetching 1 URL(s) for facebook
2026-04-13 21:16:31.918 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:130 - Using platform scraper: facebook
2026-04-13 21:16:32.263 | INFO     | syft_ingest.sources.brightdata:fetch_async:152 - Triggering facebook scrape for https://www.facebook.com/profile.php?id=61583734012155
2026-04-13 21:16:32.668 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:158 - Scrape job created: sd_mny1v9xx2cezhsm7sc
2026-04-13 21:16:32.670 | INFO     | syft_ingest.sources.brightdata:fetch_async:161 - Polling job sd_mny1v9xx2cezhsm7sc with timeout=180s, poll_interval=5s
2026-04-13 21:18:25.318 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:172 - Job sd_mny1v9xx2cezhsm7sc completed
2026-04-13 21:18:26.143 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:176 - Fetched 162690 bytes from job
2026-04-13 21:18:26.144 | WARNING  | syft_ingest.sources.brightdata:_p

Items fetched: 25
Author: Syft Influencer Test

First item:
  Type: SocialPostResult
  Title: 122103859383124467
  Text: Private Deep Learning for Hospitals using Federated Learning and Differential Privacy

Today, Kritika Prakash, Lucile Sa


## 6. Instagram scraping with posts_limit for testing

In [ ]:
corpus = await async_gather(
    "instagram",
    ["https://www.instagram.com/syft_influencer_test/"],
    author="Syft Influencer Test",
    timeout=120,
    poll_interval=5,
    posts_limit=5,  # Limit to 5 posts for quick testing (no posts_limit = all posts)
)

print(f"Items fetched: {len(corpus.all_items())}")
for i, item in enumerate(corpus.all_items()[:3], 1):
    print(f"\n{i}. {type(item).__name__}: {item.title}")

## 7. Error handling and validation

In [ ]:
# Invalid platform raises ValueError
try:
    corpus = gather("tiktok", ["https://tiktok.com/user"])
except KeyError as e:
    print(f"✅ ValueError caught for unsupported platform: {e}")

# Missing URLs raises ValueError
try:
    corpus = gather("youtube", None)
except ValueError as e:
    print(f"✅ ValueError caught for missing URLs: {e}")

## 8. Configuration validation with FetchConfig

In [ ]:
# FetchConfig provides validation and IDE autocomplete
# All these options are validated:

config = FetchConfig(
    # YouTube options
    socket_timeout=60,  # Network timeout (seconds)
    playlistend=10,  # Max videos from channel
    download_full_video=False,  # Enable full video download
    # BrightData options
    timeout=300,  # Scrape job timeout
    poll_interval=2,  # Check frequency
    posts_limit=5,  # Limit posts (for testing)
)

print("✅ FetchConfig created with validation:")
print(f"  socket_timeout: {config.socket_timeout}")
print(f"  posts_limit: {config.posts_limit}")
print()

# Invalid values are caught immediately
try:
    bad = FetchConfig(socket_timeout=-1)  # Must be >= 1
except Exception as e:
    print(f"✅ Validation error (expected): {type(e).__name__}")

In [ ]:
# Simplest: just platform + URLs
corpus = si.gather("youtube", ["https://youtube.com/watch?v=..."])

# With author
corpus = si.gather("youtube", ["https://youtube.com/watch?v=..."], author="Andrej")

# With config options
corpus = si.gather(
    "youtube", ["https://youtube.com/@user"], socket_timeout=60, playlistend=10
)

# With Instagram/Facebook (requires BRIGHTDATA_API_TOKEN)
corpus = si.gather(
    "instagram", ["https://instagram.com/user/"], timeout=120, posts_limit=5
)

# Export results
# corpus.export("jsonl", output="out.jsonl")

print("✅ API ready for use!")